# Notebook 4: Sensitivity Flagging — Method 1 (Semantic Similarity)

This notebook classifies images for sensitive content using **semantic similarity** between LLaVA-generated image descriptions and a predefined sensitive-content taxonomy.

### Pipeline
1. Load taxonomy CSV (labels + keywords)
2. Load image descriptions CSV (short + natural descriptions from LLaVA)
3. Embed taxonomy keywords using `SentenceTransformer` (`all-mpnet-base-v2`)
4. Compute cosine similarity between combined image descriptions and keyword embeddings
5. Flag images where similarity ≥ 0.45 against any category
6. Export results with matched labels, keywords, and detailed JSON

### Performance
- ~9 minutes for 1,000 images
- Generating natural descriptions: ~4 min/image; strict one-sentence prompts: ~1.5 min/image

### Prerequisites
- Run Notebook 3 first to generate image descriptions
- `pip install sentence-transformers torch`

## Configuration

In [ ]:
# ============================================================
# CONFIGURATION — Edit these paths before running
# ============================================================

# CSV with taxonomy labels and keywords
TAXONOMY_PATH = r"path/to/Sensitive_Content_Taxonomy_DataFrame.csv"

# CSV with image descriptions (from LLaVA output)
# Expected columns: file_path, image_description (strict), image_description_natural (long)
DESCRIPTIONS_PATH = r"path/to/image_descriptions.csv"

# Output CSV path
OUTPUT_PATH = r"path/to/output/semantic_sensitivity_output.csv"

# Cosine similarity threshold — images scoring >= this are flagged
# Lower = more sensitive (more false positives); Higher = more strict (more false negatives)
SIMILARITY_THRESHOLD = 0.45

print(f"Taxonomy:     {TAXONOMY_PATH}")
print(f"Descriptions: {DESCRIPTIONS_PATH}")
print(f"Output:       {OUTPUT_PATH}")
print(f"Threshold:    {SIMILARITY_THRESHOLD}")

## Imports

In [ ]:
import re
import ast
import json
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch
from tqdm import tqdm

## Load Data

In [ ]:
taxonomy_df = pd.read_csv(TAXONOMY_PATH)
flag_df = pd.read_csv(DESCRIPTIONS_PATH)

print(f"Taxonomy rows:    {len(taxonomy_df)}")
print(f"Images to check:  {len(flag_df)}")
print(f"\nTaxonomy labels: {list(taxonomy_df['Label'])}")

## Parse Taxonomy Keywords

In [ ]:
def parse_keywords(cell) -> list:
    """Parse a keywords cell — handles lists, dicts, or comma/semicolon-delimited strings."""
    if pd.isna(cell):
        return []
    s = str(cell).strip()
    try:
        val = ast.literal_eval(s)
        if isinstance(val, (list, tuple)):
            return [str(x).strip() for x in val if str(x).strip()]
        if isinstance(val, dict) and "keywords" in val:
            return [str(x).strip() for x in val["keywords"] if str(x).strip()]
    except Exception:
        pass
    return [w.strip() for w in re.split(r"[;,]", s) if w.strip()]


taxonomy = []
for _, row in taxonomy_df.iterrows():
    label = str(row.get("Label", "")).strip()
    keywords = parse_keywords(row.get("Keywords", ""))
    if label and keywords:
        taxonomy.append({"label": label, "keywords": keywords})

print(f"Parsed {len(taxonomy)} taxonomy entries.")
for t in taxonomy:
    print(f"  {t['label']}: {len(t['keywords'])} keywords")

## Embed Taxonomy Keywords

In [ ]:
print("Loading SentenceTransformer model (all-mpnet-base-v2)...")
model = SentenceTransformer("all-mpnet-base-v2")

print("\nEncoding taxonomy keywords...")
for item in tqdm(taxonomy):
    norm_keywords = [kw.lower().strip() for kw in item["keywords"]]
    item["keyword_embeddings"] = model.encode(
        norm_keywords, convert_to_tensor=True, show_progress_bar=False
    )

print("Taxonomy embeddings ready.")

## Combine Image Descriptions & Run Matching

In [ ]:
def combine_desc(short, long_desc) -> str:
    """Concatenate short and natural descriptions for richer context."""
    parts = []
    if isinstance(short, str) and short.strip():
        parts.append(short.strip())
    if isinstance(long_desc, str) and long_desc.strip():
        parts.append(long_desc.strip())
    return " ".join(parts).strip()


flag_df["combined_description"] = flag_df.apply(
    lambda r: combine_desc(r.get("image_description"), r.get("image_description_natural")),
    axis=1,
)


def match_sensitive_semantic(text: str, threshold: float = SIMILARITY_THRESHOLD) -> dict:
    """Compare a text embedding against all taxonomy keyword embeddings."""
    if not isinstance(text, str) or not text.strip():
        return {"any_sensitive": False, "matches": [], "matched_labels": [], "matched_keywords": []}

    text_emb = model.encode(text.lower().strip(), convert_to_tensor=True, show_progress_bar=False)

    matched = []
    matched_labels = set()
    matched_keywords = set()

    for item in taxonomy:
        cos_sim = util.cos_sim(text_emb, item["keyword_embeddings"])[0]
        indices = (cos_sim >= threshold).nonzero(as_tuple=True)[0]
        if len(indices) > 0:
            hits = [item["keywords"][i] for i in indices]
            matched.append({"label": item["label"], "keywords": hits})
            matched_labels.add(item["label"])
            matched_keywords.update(hits)

    return {
        "any_sensitive": bool(matched),
        "matches": matched,
        "matched_labels": sorted(matched_labels),
        "matched_keywords": sorted(matched_keywords),
    }


print("Applying semantic matching...")
tqdm.pandas()
results = flag_df["combined_description"].progress_apply(
    lambda x: match_sensitive_semantic(x, threshold=SIMILARITY_THRESHOLD)
)

flag_df["sensitive_any"] = results.apply(lambda x: "Yes" if x["any_sensitive"] else "No")
flag_df["sensitive_labels"] = results.apply(lambda x: "; ".join(x["matched_labels"]))
flag_df["sensitive_keywords"] = results.apply(lambda x: "; ".join(x["matched_keywords"]))
flag_df["sensitive_detail_json"] = results.apply(lambda x: json.dumps(x["matches"], ensure_ascii=False))

flagged_count = (flag_df["sensitive_any"] == "Yes").sum()
print(f"\nFlagged {flagged_count} / {len(flag_df)} images as potentially sensitive.")

## Export Results

In [ ]:
OUTPUT_COLUMNS = [
    "file_path",
    "image_description",
    "image_description_natural",
    "combined_description",
    "sensitive_any",
    "sensitive_labels",
    "sensitive_keywords",
    "sensitive_detail_json",
]

# Only keep columns that exist in the dataframe
cols_to_export = [c for c in OUTPUT_COLUMNS if c in flag_df.columns]
out_df = flag_df[cols_to_export].copy()
out_df.to_csv(OUTPUT_PATH, index=False)

print(f"Results saved to: {OUTPUT_PATH}")
out_df[out_df["sensitive_any"] == "Yes"].head()